<h1 style="color:black; font-family:Arial, sans-serif;">🏡 Scraping Immobilier – Avito.ma</h1>

<hr style="border:1px solid #ccc;">

<h2 style="color:#117A65;">Data Transformation</h2>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import re
from scipy import stats
import mysql.connector

In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="avito_db"
)

cursor = conn.cursor()

In [ ]:
query = "SELECT * FROM Avito_Bureaux"
df = pd.read_sql(query, conn)
df.head()

In [ ]:
df.shape[0]

In [ ]:
print("The different Features are : ",df.columns)

In [ ]:
# Replace 'Nan' strings with actual NaN values
df = df.replace('Nan', np.nan)

In [ ]:
df = df[~df['Surface_Totale'].isna()]
print(df.shape[0])

In [ ]:
df.head()

In [ ]:
import warnings

warnings.filterwarnings('ignore')


In [ ]:
df['Ville'].unique()

In [ ]:
df['Ville'] = df['Ville'].replace('الدار البيضاء', 'Casablanca').replace('المحمدية', 'Mohammedia')
df['Ville'].unique()

In [ ]:
# Calculate the number of nulls in each column
nulls_per_column = df.isnull().sum()

# Calculate the total number of nulls in the DataFrame
total_nulls = nulls_per_column.sum()

# Calculate the total number of unique records that contain at least one null value
total_unique_records_with_nulls = len(df[df.isnull().any(axis=1)].drop_duplicates())

print("Nulls in each feature:")
print(nulls_per_column)
print("\nTotal nulls in the DataFrame:", total_nulls)
print("\nTotal records with at least one null value: ", total_unique_records_with_nulls)

In [ ]:
df.drop(['Surface_Habitable', 'Chambres', 'Salons', 'Standing'], axis=1, inplace=True)

In [ ]:
df.columns

In [ ]:
ville_counts = df['Ville'].value_counts()
print(ville_counts)

In [ ]:
df['Ville'] = df['Ville'].replace('Rabat', 'Rabat-Temara').replace('Temara', 'Rabat-Temara').replace('Fès', 'Fès-Meknès').replace('Meknès', 'Fès-Meknès')

In [ ]:
seuil = 20

# Compter les occurrences de chaque ville
ville_counts = df['Ville'].value_counts()

# Villes fréquentes = celles au-dessus du seuil
villes_frequentes = ville_counts[ville_counts >= seuil].index

# Remplacer les villes rares par 'Autres'
df['Ville'] = df['Ville'].apply(lambda x: x if x in villes_frequentes else 'Autres')

# Vérifier le résultat
print(df['Ville'].value_counts())

In [ ]:
df['Etage'].value_counts()

In [ ]:
df['Age'].value_counts()

In [ ]:
df['Age_encoded'] = df['Age'].map({'Moins de 1 an': 0, '1-5 ans': 1, '6-10 ans': 2, '11-20 ans': 3, '21+ ans': 4})
df.head()

In [ ]:
df['Condition'].value_counts()

In [ ]:
df['Condition_encoded'] = df['Condition'].map({'Neuf':0, 'Bon état':1, 'à rénover':2})
df.head()

In [ ]:
df_sorted = df.sort_values(by='Prix', ascending=True)
df_sorted = df_sorted[['Prix', 'Lien']]
# Afficher les 10 premières lignes triées par prix
print(df_sorted.head(10))

In [ ]:
df['Salles_De_Bain'].value_counts()

In [ ]:
# Function to convert a single price to Dirhams
def convert_to_dirhams(price):
    if isinstance(price, str):
        # Replace commas with periods and non-breaking spaces with nothing
        price = price.replace(',', '.').replace('\xa0', '')
        # Remove ' DH' and convert to float
        return float(price.replace('DH', '').replace(' ', ''))
    else:
        # If not a string, return as is
        return price

# Apply the function to each value in the 'Prix' column
df['Prix'] = df['Prix'].apply(convert_to_dirhams)

# Convert the non-null values in the 'Prix' column to integer
df.loc[df['Prix'].notna(), 'Prix'] = df.loc[df['Prix'].notna(), 'Prix'].astype(int)

df.head()

In [ ]:
columns = ['Terrasse', 'Parking', 'Securite', 'Chauffage', 'Climatisation', 'Ascenseur']
for column in columns:
    df[column] = df[column].replace({"oui": 1, "non": 0})
    
df.head()

In [ ]:
df.info()

In [ ]:

df['Age_encoded'] = df.groupby('Ville')['Age_encoded'].transform(
    lambda x: x.fillna(x.median())
)
df['Age_encoded'] = df['Age_encoded'].fillna(df['Age_encoded'].median())
df['Age_encoded'] = df['Age_encoded'].round().astype(int)


In [ ]:
df['Age_encoded'].unique()

In [ ]:
df['Condition_encoded'] = df.groupby('Ville')['Condition_encoded'].transform(
    lambda x: x.fillna(x.median())
)
df['Condition_encoded'] = df['Condition_encoded'].fillna(df['Condition_encoded'].median())
df['Condition_encoded'] = df['Condition_encoded'].round().astype(int)

In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')
df['Salles_De_Bain'] = num_imputer.fit_transform(df[['Salles_De_Bain']])
df['Etage'] = num_imputer.fit_transform(df[['Etage']])


In [ ]:
df.info()

In [ ]:
df['Surface_Totale'] = pd.to_numeric(df['Surface_Totale'], errors='coerce')
df['Surface_Totale'] = df['Surface_Totale'].astype(float)

In [ ]:
# quartiers_par_ville_df = quartiers_par_ville.reset_index()
# quartiers_par_ville_df.columns = ['Ville', 'Quartiers']
# for ville, quartiers in quartiers_par_ville.items():
#     print(f"Ville: {ville}")
#     print(f"Quartiers:\n {list(quartiers)}\n")

In [ ]:
from rapidfuzz import fuzz, process

# Exemple : liste unique de quartiers
quartiers = df['Quartier'].unique()

# Dictionnaire de correspondance
corrections = {}

# Boucle pour comparer chaque quartier aux autres
for i, q1 in enumerate(quartiers):
    for q2 in quartiers[i+1:]:
        score = fuzz.ratio(q1.lower(), q2.lower())
        if score >= 85:  # seuil de similarité
            print(f"{q1} <--> {q2} : {score}% similaire")
            # On choisit la forme la plus courte comme forme standard
            standard = min(q1, q2, key=len)
            corrections[q1] = standard
            corrections[q2] = standard

print("\nDictionnaire de remplacement :")
print(corrections)


In [ ]:
# Fonction de remplacement
def corriger_quartier(q):
    return corrections.get(q, q)  # si pas dans le dict, garder original

df['Quartier'] = df['Quartier'].apply(corriger_quartier)


In [ ]:
# Calculer la moyenne du prix par quartier
quartier_mean = df.groupby('Quartier')['Prix'].mean()

# Associer chaque ligne à la moyenne de son quartier
df['Quartier_enc'] = df['Quartier'].map(quartier_mean)

In [ ]:
df.head()

In [ ]:
df = df[(df['Prix'] <= 6000000) & (df['Prix']>= 50000)]
df.shape[0]

<h2 style="color:black"> Using IQR </h2>

In [ ]:
df["Salles_De_Bain"] = pd.to_numeric(df["Salles_De_Bain"], errors="coerce")
Q1 = df["Salles_De_Bain"].quantile(0.25)
Q3 = df["Salles_De_Bain"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition = (df["Salles_De_Bain"] < (Q1 - 1.5 * IQR)) | (df["Salles_De_Bain"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliers = df[outlier_condition]["Salles_De_Bain"]

# Print the number of outliers
print("\nNumber of outliers in Nombre De Salles_De_Bain:", outliers.count())

In [ ]:
median_sdb = df.loc[~outlier_condition, 'Salles_De_Bain'].median()

df.loc[outlier_condition, 'Salles_De_Bain'] = median_sdb

# Arrondir et forcer entier
df['Salles_De_Bain'] = df['Salles_De_Bain'].round().astype(int)

In [ ]:
df['Salles_De_Bain'].value_counts()

In [ ]:
Q1 = df["Prix"].quantile(0.25)
Q3 = df["Prix"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition = (df["Prix"] < (Q1 - 1.5 * IQR)) | (df["Prix"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliersQP = df[outlier_condition]

# Print the number of outliers
print("\nNumber of outliers in Prix:", outliersQP.shape[0])

In [ ]:
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['Prix'] = df['Prix'].clip(lower=lower_bound, upper=upper_bound)


In [ ]:
Q1 = df["Surface_Totale"].quantile(0.25)
Q3 = df["Surface_Totale"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition1 = (df["Surface_Totale"] < (Q1 - 1.5 * IQR)) | (df["Surface_Totale"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliersQS = df[outlier_condition1]
# Print the number of outliers
print("\nNumber of outliers in Surface:", outliersQS.shape[0])

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['Surface_Totale'] = df['Surface_Totale'].clip(lower=lower_bound, upper=upper_bound)


<h2 style="color:black"> Using z-score </h2>

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6)) 
plot = sns.histplot(df['Surface_Totale'], binwidth=15, color='orange', ax=ax)
ax.set_xlabel('Surface_Totale')
ax.set_ylabel('Frequency')
ax.set_xlim(0, 500)
ax.set_aspect('equal')  
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='Surface_Totale', y='Prix', color='#0080ff')

plt.title("Prix vs Surface ")
plt.xlabel("Surface")
plt.ylabel("Prix")
plt.xlim(0, 500)
plt.ylim(0, 6000000)

plt.tight_layout()
plt.show()


In [ ]:
outliersQP.describe()

In [ ]:
outliersQS.describe()

In [ ]:
plt.figure(figsize=(6.6, 4.6))
plt.scatter( df['Surface_Totale'] ,df['Prix'] ,color='#0080ff', label='Training Data')
plt.xlabel("Surface_Totale")
plt.ylabel('Prix')
plt.xlim(0, 500)
plt.ylim(0, 6000000)
plt.show()

In [ ]:
df.columns

In [ ]:
# villes = ['Ville_Agadir', 'Ville_Autres', 'Ville_Casablanca', 'Ville_Fès-Meknès', 'Ville_Kénitra',
#        'Ville_Marrakech', 'Ville_Mohammedia', 'Ville_Oujda', 'Ville_Rabat-Temara', 'Ville_Tanger']

# for ville in villes:
#     df[ville] = df[ville].map({True: 1, False: 0})


In [ ]:
df.columns

In [ ]:
import numpy as np

# Remplacer les vrais NaN (np.nan) ou chaînes "nan"/"NaN" par None
df = df.replace(["nan", "NaN", np.nan], None)

# Liste des colonnes sans 'id'
columns = [col for col in df.columns if col != "id"]

# Construction de la requête
cols_str = ", ".join(f"`{col}`" for col in columns)
placeholders = ", ".join(["%s"] * len(columns))
insert_query = f"INSERT INTO bureaux_Transformed ({cols_str}) VALUES ({placeholders})"

# Insertion dans MySQL
for _, row in df[columns].iterrows():
    cursor.execute(insert_query, tuple(row))

conn.commit()


In [ ]:
cursor.close()
conn.close()